In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
import eodgdl
from eodgdl import giro
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rc("text", usetex=True)
mpl.rc("font", family="serif")

# Economic activity (giro) imputation for OD workers

A substantial share of OD workers did not report the activity of the business they work for (`giro_empresa`). This stage estimates

$$P(G = g \mid X)$$

for the five native answers of the survey — Comercio, Servicio, Educación, Industria, Gobierno/sector público — from OD-only predictors, and keeps the whole probability vector rather than a hard assignment.

The model lives in `eodgdl.giro` (extra `eodgdl[giro]`): it reads the cleaned survey directly through `eodgdl.load_eod()`, uses the raw survey columns (no ENOE harmonization), attaches the work-trip destination and its DENUE establishment mix (via `mxcensus`), and predicts the survey's own giro levels. Downstream users collapse or relabel the five probabilities as they need; the informal-jobs-model pipeline, for instance, maps them to four ENOE-comparable sector classes.

In [ ]:
od = giro.build_worker_features(eodgdl.load_eod())
print(f"OD workers: {len(od):,}")
print(f"Workers with known giro: {(~od['giro_desconocido']).sum():,}")
print(f"Workers with unknown giro: {od['giro_desconocido'].sum():,}")
print(od["giro"].value_counts(dropna=False).rename(index=giro.GIRO_LABELS).to_string())

## Giro missingness profiles

Weighted characteristics of workers with and without an observed giro: the training population differs from the imputation target, which motivates the covariate-shift sensitivity check at the end.

In [ ]:
profile_columns = ["sexo_nacimiento", "ocupacion", "escolaridad", "municipio", "estado_civil", "parentesco", "personas_en_vivienda", "n_autos_camionetas"]
missingness_profiles = giro.compare_known_unknown_profiles(od, profile_columns)

fig, axes = plt.subplots(2, 4, figsize=(22, 9))
for i, (ax, variable) in enumerate(zip(axes.flatten(), profile_columns)):
    plot_data = missingness_profiles[missingness_profiles["variable"] == variable].set_index("category")[["known_share", "unknown_share"]] * 100
    plot_data.index = [str(label)[:18] for label in plot_data.index]
    plot_data.plot(kind="bar", ax=ax, color=[(0.06, 0.31, 0.55), (0.33, 0.53, 0.55)])
    ax.set_title(variable.replace("_", " "), fontsize=14); ax.set_xlabel("")
    ax.set_ylabel(r"Proporción ponderada [\%]" if i in (0, 4) else "")
    ax.tick_params(axis="x", rotation=60, labelsize=8); ax.get_legend().remove()
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, ["Giro conocido", "Giro desconocido"], loc="lower center", ncol=2, fontsize=13, bbox_to_anchor=(0.5, 0.0))
output_dir = ROOT / "outputs/figures"; output_dir.mkdir(parents=True, exist_ok=True)
plt.tight_layout(rect=[0, 0.05, 1, 1])
fig.savefig(output_dir / "sector_known_unknown_profiles.pdf", bbox_inches="tight"); fig.savefig(output_dir / "sector_known_unknown_profiles.png", dpi=600, bbox_inches="tight")
plt.show()

## Predictors

Two specifications: with education (`SECTOR_FEATURES`) and without (`ROBUST_SECTOR_FEATURES`, for the workers whose education is unobserved). Besides the person and dwelling attributes, the model uses OD-specific information without ENOE counterparts: the raw occupation and employment status, the dwelling's centrality, the work trip (destination type, mode, weekend work travel, household vehicles) and the destination's ámbito and DENUE establishment mix.

In [ ]:
with_education_features = giro.SECTOR_FEATURES
without_education_features = giro.ROBUST_SECTOR_FEATURES
print("With education:", *with_education_features, sep="\n  ")
print("\nWithout education drops:", sorted(set(with_education_features) - set(without_education_features)))
giro.calculate_feature_missingness(od, features=with_education_features).round(4)

## Household-grouped train-test split

Workers with an observed giro are split 80/20 by household (`StratifiedGroupKFold`, fold 0 held out), so members of the same dwelling never straddle the split.

In [ ]:
train, test = giro.split_known_data(od, n_splits=5, test_fold=0, random_state=42)
print(f"Training workers: {len(train):,} in {train['folio_vivienda'].nunique():,} households")
print(f"Test workers: {len(test):,} in {test['folio_vivienda'].nunique():,} households")

## Model A: with education

Three families are tuned under a 5-fold household-grouped CV with expansion-factor weights; within each family, and then across families, the simplest configuration within one standard error of the best log loss is selected.

In [ ]:
from IPython.display import Markdown
models = giro.build_models()
lines = [f"* **{name}** — tuned: " + ", ".join(f"`{key.replace('classifier__', '')}`: {values}" for key, values in config["params"].items()) for name, config in models.items()]
total = sum(int(__import__("math").prod(len(v) for v in c["params"].values())) for c in models.values())
display(Markdown("\n".join(lines) + f"\n\n{total} configurations × 5 folds = {total * 5} fits per specification."))

In [ ]:
X_a, y_a, w_a, g_a, _ = giro.prepare_training_data(train, features=with_education_features)
with_education_summary, with_education_models = giro.tune_models(X_a, y_a, w_a, g_a, features=with_education_features, cv_splits=5, random_state=42)
with_education_summary

In [ ]:
with_education_best_name, with_education_best_model = giro.get_best_model(with_education_summary, with_education_models)
print(f"Selected: {with_education_best_name}", with_education_summary.loc[with_education_summary["selected"], "best_params"].iloc[0])
giro.fold_table(with_education_summary).round(4)

In [ ]:
with_education_metrics, with_education_class_metrics, with_education_confusion, with_education_distribution = giro.evaluate_model(with_education_best_model, test, features=with_education_features)
with_education_uncertainty = giro.test_metrics_with_uncertainty(with_education_best_model, test, features=with_education_features)
display(with_education_metrics.round(4)); display(with_education_uncertainty.round(4)); display(with_education_class_metrics.round(4)); display(with_education_distribution.round(4))

In [ ]:
from IPython.display import Markdown
m = with_education_metrics.iloc[0]; u = with_education_uncertainty; cm = with_education_class_metrics.set_index("giro"); rare = cm["recall"].idxmin()
d = with_education_distribution.set_index("giro"); prob_gap = (d["probabilistic_predicted_share"] - d["observed_share"]).abs().max() * 100; hard_gap = (d["hard_predicted_share"] - d["observed_share"]).abs().max() * 100
display(Markdown(f"The selected **{with_education_best_name}** reaches a weighted accuracy of {m['weighted_accuracy']:.1%} on the held-out households (95% household-bootstrap interval {u.loc['weighted_accuracy', 'ci_low']:.1%}–{u.loc['weighted_accuracy', 'ci_high']:.1%}) and a weighted log loss of {m['weighted_log_loss']:.4f}, {u.loc['relative_improvement_over_marginal', 'estimate']:.1%} below the weighted-marginal baseline ({u.loc['marginal_log_loss', 'estimate']:.4f}). "
    f"Performance is uneven across classes: `{rare}` (recall {cm.loc[rare, 'recall']:.1%}) is rarely selected by the hard classifier. Hard assignments distort the aggregate composition by up to {hard_gap:.1f} pp, whereas the probabilistic aggregation stays within {prob_gap:.1f} pp of the observed shares — the reason to keep the full probability vector."))

## Model B: without education

In [ ]:
X_b, y_b, w_b, g_b, _ = giro.prepare_training_data(train, features=without_education_features)
without_education_summary, without_education_models = giro.tune_models(X_b, y_b, w_b, g_b, features=without_education_features, cv_splits=5, random_state=42)
without_education_best_name, without_education_best_model = giro.get_best_model(without_education_summary, without_education_models)
print(f"Selected: {without_education_best_name}", without_education_summary.loc[without_education_summary["selected"], "best_params"].iloc[0])
display(without_education_summary); display(giro.fold_table(without_education_summary).round(4))

In [ ]:
without_education_metrics, without_education_class_metrics, without_education_confusion, without_education_distribution = giro.evaluate_model(without_education_best_model, test, features=without_education_features)
without_education_uncertainty = giro.test_metrics_with_uncertainty(without_education_best_model, test, features=without_education_features)
display(without_education_metrics.round(4)); display(without_education_uncertainty.round(4)); display(without_education_class_metrics.round(4)); display(without_education_distribution.round(4))

In [ ]:
from IPython.display import Markdown
a = with_education_metrics.iloc[0]; b = without_education_metrics.iloc[0]
display(Markdown(f"Without education, weighted accuracy moves from {a['weighted_accuracy']:.1%} to {b['weighted_accuracy']:.1%} and the weighted log loss from {a['weighted_log_loss']:.4f} to {b['weighted_log_loss']:.4f} ({without_education_uncertainty.loc['relative_improvement_over_marginal', 'estimate']:.1%} below the marginal baseline instead of {with_education_uncertainty.loc['relative_improvement_over_marginal', 'estimate']:.1%}). Model B is a reasonable fallback for the workers whose education is unobserved."))
model_comparison = pd.concat([with_education_metrics.assign(features="Con escolaridad"), without_education_metrics.assign(features="Sin escolaridad")], ignore_index=True)
model_comparison[["features", "weighted_accuracy", "weighted_balanced_accuracy", "weighted_f1_macro", "weighted_log_loss"]].round(4)

### Calibration of the giro probabilities

One-vs-rest reliability of each class on the held-out households: the imputation is probabilistic, so what matters is whether the probabilities are honest.

In [ ]:
with_education_calibration, with_education_reliability = giro.calculate_calibration(with_education_best_model, test, features=with_education_features)
without_education_calibration, without_education_reliability = giro.calculate_calibration(without_education_best_model, test, features=without_education_features)
print("Calibration in the large — with education"); print(with_education_calibration.round(4).to_string(index=False))
print("\nCalibration in the large — without education"); print(without_education_calibration.round(4).to_string(index=False))

fig, axes = plt.subplots(2, 3, figsize=(14, 9), sharex=True, sharey=True)
for ax, giro in zip(axes.ravel(), giro.GIRO_CLASSES):
    ax.plot([0, 1], [0, 1], linestyle="--", color="black", linewidth=1)
    for label, reliability, color in (("Con escolaridad", with_education_reliability, (0.06, 0.31, 0.55)), ("Sin escolaridad", without_education_reliability, (0.36, 0.67, 0.93))):
        curve = reliability[reliability["giro"] == giro]
        ax.plot(curve["predicted_probability"], curve["observed_rate"], marker="o", color=color, label=label)
        ax.scatter(curve["predicted_probability"], curve["observed_rate"], s=curve["sample_workers"] / 5, color=color, alpha=0.3)
    ax.set_title(giro.GIRO_LABELS[giro], fontsize=13); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
axes[1, 2].axis("off"); axes[0, 0].legend(fontsize=9)
for ax in axes[1, :2]: ax.set_xlabel("Probabilidad predicha")
for ax in axes[:, 0]: ax.set_ylabel("Proporción observada")
fig.suptitle("Calibración de las probabilidades de giro (hogares de prueba)", fontsize=14)
plt.tight_layout()
fig.savefig(output_dir / "sector_calibration.pdf", bbox_inches="tight"); fig.savefig(output_dir / "sector_calibration.png", dpi=600, bbox_inches="tight")
plt.show()

## Hybrid model: refit and imputation

Both selected specifications are refitted on every worker with an observed giro. Workers without a giro are scored by Model A when their education is observed and by Model B otherwise. Workers without a work trip on the survey day have no destination: they are marginalized over destination types with their own conditional distribution P(destino | x) from an auxiliary model, and any categorical level without training support is marginalized over the supported levels (`giro_marginalized_features` records where this happened).

In [ ]:
final_with_education_model, final_training = giro.refit_model(with_education_best_model, od, features=with_education_features)
final_without_education_model, _ = giro.refit_model(without_education_best_model, od, features=without_education_features)
destination_models = giro.fit_destination_models(od, with_education_features=with_education_features, without_education_features=without_education_features)
od_giro = giro.impute_giro(final_with_education_model, final_without_education_model, od, with_education_features=with_education_features, without_education_features=without_education_features, destination_models=destination_models)
print(f"Final training workers: {len(final_training):,}; imputed workers: {od_giro['giro_fue_imputado'].sum():,}; without final giro: {od_giro['giro_final'].isna().sum():,}")
print(f"Maximum probability-sum error: {giro.validate_probability_rows(od_giro):.3e}")
display(giro.calculate_model_usage(od_giro).round(4)); display(giro.calculate_confidence_summary(od_giro).round(4))
print("Imputed rows with categorical levels absent from the training data:")
display(giro.count_levels_without_training_support(giro.prepare_model_features(od[~od["giro_desconocido"]], with_education_features), giro.prepare_model_features(od[od["giro_desconocido"]], with_education_features)))

In [ ]:
imputed_workers = od_giro[od_giro["giro_fue_imputado"]]
observed_distribution = giro.calculate_distribution(od[~od["giro_desconocido"]], giro_column="giro")
hard_imputed = giro.calculate_distribution(imputed_workers); probabilistic_imputed = giro.calculate_probabilistic_distribution(imputed_workers)
hard_final = giro.calculate_distribution(od_giro); probabilistic_final = giro.calculate_probabilistic_distribution(od_giro)

comparison = observed_distribution[["giro", "weighted_share"]].rename(columns={"weighted_share": "observed_known_share"})
comparison = comparison.merge(probabilistic_imputed[["giro", "weighted_share"]].rename(columns={"weighted_share": "imputed_share"}), on="giro", how="outer")
comparison = comparison.merge(hard_imputed[["giro", "weighted_share"]].rename(columns={"weighted_share": "hard_imputed_share"}), on="giro", how="outer")
comparison = comparison.merge(probabilistic_final[["giro", "weighted_share"]].rename(columns={"weighted_share": "final_share"}), on="giro", how="outer").set_index("giro").reindex(giro.GIRO_CLASSES)
display((comparison * 100).round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
plot_data = comparison[["observed_known_share", "imputed_share", "final_share"]] * 100
plot_data.index = [giro.GIRO_LABELS[g].replace("/", "/\n") for g in plot_data.index]
plot_data.plot(kind="bar", ax=axes[0], color=[(0.06, 0.31, 0.55), (0.33, 0.53, 0.55), (0.36, 0.67, 0.93)])
axes[0].set_title("Distribución por giro", fontsize=15); axes[0].set_xlabel(""); axes[0].set_ylabel(r"Proporción ponderada [\%]"); axes[0].tick_params(axis="x", rotation=0, labelsize=10)
axes[0].legend(["Conocido", "Asignado (prob.)", "OD final"], fontsize=10)
for key, color, label in (("without_education", (0.06, 0.31, 0.55), "Sin escolaridad"), ("with_education", (0.33, 0.53, 0.55), "Con escolaridad")):
    axes[1].hist(imputed_workers.loc[imputed_workers["giro_model_used"] == key, "giro_prediction_confidence"], bins=20, alpha=0.5, color=color, label=label)
axes[1].set_title("Confianza de predicción", fontsize=15); axes[1].set_xlabel("Confianza"); axes[1].set_ylabel("Trabajadores"); axes[1].legend()
plt.tight_layout()
fig.savefig(output_dir / "sector_final_distributions_confidence.pdf", bbox_inches="tight"); fig.savefig(output_dir / "sector_final_distributions_confidence.png", dpi=600, bbox_inches="tight")
plt.show()

### Sensitivity to the covariate shift

The model is validated on known-giro households but applied to the unknown-giro workers, who differ (more educated, more item non-response). Two checks: (i) refit the selected models on known-giro rows reweighted to the unknown-giro profile (density-ratio weights on the shift profile features, education excluded); (ii) a delta adjustment that scales the imputed probability of the minority class to the observed share. Both scenarios are saved for downstream sensitivity tables.

In [ ]:
od_giro_shift, shift_diagnostics = giro.impute_under_covariate_shift(with_education_best_model, without_education_best_model, od, with_education_features=with_education_features, without_education_features=without_education_features)
od_giro_delta, delta_factor = giro.adjust_imputed_share(od_giro, "gobierno")
print("Shift-weighting diagnostics:"); print(shift_diagnostics.round(3).to_string())
print(f"\nDelta adjustment factor for gobierno among imputed workers: {delta_factor:.3f}")
imputed_shares = lambda frame: giro.calculate_probabilistic_distribution(frame[frame["giro_fue_imputado"]]).set_index("giro")["weighted_share"]
sensitivity = pd.DataFrame({"observed (known giro)": observed_distribution.set_index("giro")["weighted_share"], "imputed: current": imputed_shares(od_giro), "imputed: shift-weighted": imputed_shares(od_giro_shift), "imputed: delta-adjusted": imputed_shares(od_giro_delta)}).reindex(giro.GIRO_CLASSES)
print("\nGiro shares among imputed workers (weighted, %):"); print((sensitivity * 100).round(2))

## Outputs

`outputs/od_giro_imputed.parquet` holds the scored workers (keys, `giro_*` columns and `prob_giro_<slug>`; `giro.OUTPUT_COLUMNS`) and the two sensitivity scenarios go to `od_giro_imputed_sensitivity.parquet`. The fitted bundle is written to `data/od_giro_hybrid_model.joblib`, from where the package serves it (`giro.load_model()`); after retraining, update its sha256 in `src/eodgdl/data/registry.txt`.

In [ ]:
output_directory = ROOT / "outputs"
output_directory.mkdir(exist_ok=True)
od_giro[giro.OUTPUT_COLUMNS].to_parquet(output_directory / "od_giro_imputed.parquet", index=False)
sensitivity_output = pd.concat([frame[giro.OUTPUT_COLUMNS].assign(scenario=name) for name, frame in (("shift_weighted", od_giro_shift), ("delta_adjusted", od_giro_delta))], ignore_index=True)
sensitivity_output.to_parquet(output_directory / "od_giro_imputed_sensitivity.parquet", index=False)

In [ ]:
from joblib import dump

models_directory = ROOT / "data"  # the bundle ships with the package data (pooch registry: update its sha256 in src/eodgdl/data/registry.txt)
sector_model_bundle = {
    "model_with_education": final_with_education_model,
    "model_without_education": final_without_education_model,
    "features_with_education": with_education_features,
    "features_without_education": without_education_features,
    "category_levels": giro.build_category_levels(),
    "giro_classes": giro.GIRO_CLASSES,
    "giro_labels": giro.GIRO_LABELS,
    "destination_models": destination_models,
    "metadata": {
        "selected": {"with_education": with_education_summary.loc[with_education_summary["selected"], ["model", "best_params", "weighted_log_loss"]].iloc[0].to_dict(), "without_education": without_education_summary.loc[without_education_summary["selected"], ["model", "best_params", "weighted_log_loss"]].iloc[0].to_dict()},
        "test_metrics": {"with_education": with_education_metrics.iloc[0].to_dict(), "without_education": without_education_metrics.iloc[0].to_dict()},
        "denue_release": giro.DENUE_RELEASE, "eodgdl_version": eodgdl.__version__, "sklearn_version": __import__("sklearn").__version__, "random_state": 42,
    },
}
model_path = models_directory / "od_giro_hybrid_model.joblib"
dump(sector_model_bundle, model_path)
print(f"Saved: {model_path}")